## Part 3. Data Clean Function
Based on the data analysis, we create a data cleaning function that performs the following steps for the dataset:

- Step1, Load data.
- Step2, Insert a column `subject_id` for future possible traceability.
- Step3, Set plausible ranges for each variable.
- Step4, Remove samples that contain values outside plausible ranges.
- Step5, Normalization (Min-Max scaling)

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler


def load_and_clean_data(csv_path):
    
    """
    Load, clean, and normalize the Pima Diabetes dataset.

    Steps performed:
    1. Load CSV into pandas DataFrame.
    2. Insert 'subject_id' column for traceability.
    3. Set plausible ranges of each variable, to remove physiologically impossible or extreme values.
    4. Remove samples that contain values outside plausible ranges.
    5. Normalize all feature columns using Min-Max scaling.

    Input:
    csv_path (str): Path to the CSV dataset.

    Returns:
    pd.DataFrame: Cleaned and normalized dataset with 'subject_id'.
    """
    
    # Step 1: Load data
    df = pd.read_csv(csv_path)

    # Step 2: Insert subject_id for traceability
    df.insert(0, 'subject_id', df.index)

    # Step 3: Set plausible ranges 
    
    # The ranges are based on references listed, but expanded for pathological values in diabetic patients.
    # You can change them in a more reasonable range if it is necessary. But don't forget to save a new version of cleaned data if you change the range（Just uncomment the line that saves to csv）
    
    plausible_ranges = {
        "Pregnancies": (0, np.inf),
        "Glucose": (40, 600),                # Mayo Clinic Laboratories Critical Values / Critical Results List, https://www.ncbi.nlm.nih.gov/books/NBK555976/
        "BloodPressure": (30, 120),          # https://www.nhlbi.nih.gov/health/low-blood-pressure, https://www.nhlbi.nih.gov/health/high-blood-pressure
        "SkinThickness": (5, 60),            # Anthropometric Reference Data for Children and Adults: United States,2007–2010
        "Insulin": (0, 600),                 # Roche Diagnostics. (2023). Elecsys Insulin: Method Sheet (V 4.0). 
        "BMI": (16, 70),                     # https://www.who.int/data/nutrition/nlis/info/malnutrition-in-women，Corpodean F, Kachmar M, Popiv I, LaPenna KB, Lenhart D, Cook M, Albaugh VL, Schauer PR. BMI ≥ 70: A Multi-Center Institutional Experience of the Safety and Efficacy of Metabolic and Bariatric Surgery Intervention. Obes Surg. 2024 Sep;34(9):3165-3172. doi: 10.1007/s11695-024-07419-7. Epub 2024 Jul 24. PMID: 39046626.
        "DiabetesPedigreeFunction": (0, np.inf),   # According to the definition of Diabetes Pedigree FunctionUsing in the paper ADAP Learning Algorithm to Forecast the Onset of Diabetes Mellitus
        "Age": (21, np.inf)
    }

    # Step 4: Remove rows outside plausible ranges
    for feature, (lower, upper) in plausible_ranges.items():
        df = df[(df[feature] >= lower) & (df[feature] <= upper)]
        

    # !!! Uncomment this line to save the cleaned dataset !!!
    # df.to_csv("diabetes_cleaned.csv", index=False)
    

    # Step 5: Data normalization (2 methods to set the features to be normalized)
    
    # Method_1: Set features that going to be normalized.
    features_to_normalize = ["Pregnancies", "Glucose", "BloodPressure", 
                           "SkinThickness", "Insulin", "BMI", 
                           "DiabetesPedigreeFunction", "Age"]
    
#     # Method_2: Drop columns that not need to be normalized.
#     features_to_normalize = df.columns.drop(["subject_id", "Outcome"])
    
    # Normalization
    scaler = MinMaxScaler() 
    df[features_to_normalize] = scaler.fit_transform(df[features_to_normalize])
    
    # Reset index after all cleaning
    df.reset_index(drop=True, inplace=True)

    return df

In [2]:
load_and_clean_data('diabetes\diabetes.csv')

,subject_id,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,0,0.352941,0.643357,0.525,0.528302,0.000000,0.314928,0.246028,0.483333,1
1,1,0.058824,0.202797,0.450,0.415094,0.000000,0.171779,0.120744,0.166667,0
2,3,0.058824,0.230769,0.450,0.301887,0.156667,0.202454,0.037222,0.000000,0
3,4,0.000000,0.566434,0.125,0.528302,0.280000,0.509202,1.000000,0.200000,1
4,6,0.176471,0.153846,0.250,0.471698,0.146667,0.261759,0.073990,0.083333,1
...,...,...,...,...,...,...,...,...,...,...
521,761,0.529412,0.797203,0.550,0.452830,0.000000,0.527607,0.144349,0.366667,1
522,763,0.588235,0.314685,0.575,0.773585,0.300000,0.300613,0.039038,0.700000,0
523,764,0.117647,0.461538,0.500,0.377358,0.000000,0.380368,0.115751,0.100000,0
524,765,0.294118,0.454545,0.525,0.301887,0.186667,0.163599,0.072628,0.150000,0
